# YOLOV8 — Benchmark on Indoor Dataset (7 classes)


In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [4]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

IS_KAGGLE = os.path.exists("/kaggle/input")

def _safe_cuda_empty_cache():
    if not torch.cuda.is_available(): return
    try: torch.cuda.synchronize()
    except: pass
    try: torch.cuda.empty_cache()
    except: pass

RUN_TRAINING = True
BENCHMARK_VARIANTS = ["yolov8n", "yolov8s", "yolov8m"]
WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

if IS_KAGGLE:
    KAGGLE_INPUT = "/kaggle/input/datasets/mariabouguettaya/dataset-indoor"
    DATA_YAML_ORIG = f"{KAGGLE_INPUT}/data.yaml"
    DATA_YAML = "/kaggle/working/data_kaggle.yaml"
    TEST_IMG_DIR = Path(f"{KAGGLE_INPUT}/test/images")
    import yaml
    if os.path.exists(DATA_YAML_ORIG):
        with open(DATA_YAML_ORIG, 'r') as f: y_data = yaml.safe_load(f)
        y_data['path'] = KAGGLE_INPUT
        y_data['train'] = "train/images"
        y_data['val'] = "valid/images"
        y_data['test'] = "test/images"
        with open(DATA_YAML, 'w') as f: yaml.dump(y_data, f)
else:
    DATA_YAML = "../data_indoor_balanced.yaml"
    TEST_IMG_DIR = Path("../dataset_indoor_yolo_new/test/images")

IMG_SIZE = 640
EPOCHS = 100
BATCH = 32
DEVICE = "0,1"
PATIENCE = 20
RESULTS_CSV  = "benchmark_yolov8_indoor.csv"
PERCLASS_CSV = "benchmark_yolov8_indoor_perclass.csv"
CLASS_NAMES = ["chair", "clock", "exit", "fireextinguisher", "printer", "screen", "trashbin"]

MODELS = [
    "yolov8n.pt",
    "yolov8s.pt",
    "yolov8m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


In [7]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df.to_string(index=False))
styled = df.style.highlight_max(subset=['mAP@0.5', 'neg_acc'], color='#2d6a2e')
display(styled)


====
  YOLOv10 BENCHMARK — OOD Dataset (22 classes)
====

    model  mAP@0.5  mAP@0.5:0.95  precision  recall     F1  speed_ms/img  size_MB  params_M
yolov10n   0.5998        0.4224     0.6797  0.5426 0.6035         14.46      5.8       2.3
yolov10s   0.6099        0.4363     0.6829  0.5653 0.6186         17.08     49.0       7.2 



model,mAP@0.5,mAP@0.5:0.95,precision,recall,F1,speed_ms/img,size_MB,params_M
yolov10n,0.5998,0.4224,0.6797,0.5426,0.6035,14.5 ms,5.8 MB,2.3 M
yolov10s,0.6099,0.4363,0.6829,0.5653,0.6186,17.1 ms,49.0 MB,7.2 M


In [8]:
df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)

print("Per-class mAP@0.5 across YOLOv8 variants")
print("-" * 60)

styled_pc = (
    df_pc.style
    .set_caption("Per-Class mAP@0.5 — YOLOv8 Benchmark")
    .format("{:.4f}")
    .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
    .set_properties(**{"text-align": "center", "font-size": "12px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
    ])
)
display(styled_pc)

Per-class mAP@0.5 across YOLOv10 variants
------------------------------------------------------------


,bench,bicycle,bus,bus_stop,car,crutch,curb,dog,fire_hydrant,motorcycle,person,pole,spherical_roadblock,stairs,stop_sign,street_light,traffic_light,train,tree,truck,warning_column,waste_container
model,,,,,,,,,,,,,,,,,,,,,,
yolov10n,0.5488,0.5174,0.7073,0.9307,0.6104,0.4733,0.3831,0.6944,0.8211,0.6371,0.2122,0.3297,0.9143,0.6255,0.9087,0.2345,0.4828,0.7675,0.1620,0.6284,0.7948,0.8116
yolov10s,0.5374,0.5309,0.7456,0.9641,0.6340,0.5869,0.4340,0.6374,0.8336,0.6127,0.1854,0.3049,0.9147,0.6009,0.9107,0.2961,0.4047,0.8008,0.1407,0.6816,0.8285,0.8309


In [9]:
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(df["model"], df["mAP@0.5"], color="#2d6a2e")
axes[0].set_xlabel("mAP@0.5")
axes[0].set_title("Accuracy (mAP@0.5)")
axes[0].set_xlim(0, 1)

axes[1].barh(df["model"], df["speed_ms/img"], color="#1a5276")
axes[1].set_xlabel("ms / image")
axes[1].set_title("Inference Speed")

axes[2].barh(df["model"], df["size_MB"], color="#c0392b")
axes[2].set_xlabel("MB")
axes[2].set_title("Model Size")

plt.suptitle("YOLOv8 Benchmark — Indoor Dataset", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("benchmark_yolov8_chart.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1800x500 with 3 Axes>

In [10]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

IS_KAGGLE = os.path.exists("/kaggle/input")

def _safe_cuda_empty_cache():
    if not torch.cuda.is_available(): return
    try: torch.cuda.synchronize()
    except: pass
    try: torch.cuda.empty_cache()
    except: pass

RUN_TRAINING = True
BENCHMARK_VARIANTS = ["yolov8n", "yolov8s", "yolov8m"]
WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

if IS_KAGGLE:
    KAGGLE_INPUT = "/kaggle/input/datasets/mariabouguettaya/dataset-indoor"
    DATA_YAML_ORIG = f"{KAGGLE_INPUT}/data.yaml"
    DATA_YAML = "/kaggle/working/data_kaggle.yaml"
    TEST_IMG_DIR = Path(f"{KAGGLE_INPUT}/test/images")
    import yaml
    if os.path.exists(DATA_YAML_ORIG):
        with open(DATA_YAML_ORIG, 'r') as f: y_data = yaml.safe_load(f)
        y_data['path'] = KAGGLE_INPUT
        y_data['train'] = "train/images"
        y_data['val'] = "valid/images"
        y_data['test'] = "test/images"
        with open(DATA_YAML, 'w') as f: yaml.dump(y_data, f)
else:
    DATA_YAML = "../data_indoor_balanced.yaml"
    TEST_IMG_DIR = Path("../dataset_indoor_yolo_new/test/images")

IMG_SIZE = 640
EPOCHS = 100
BATCH = 32
DEVICE = "0,1"
PATIENCE = 20
RESULTS_CSV  = "benchmark_yolov8_indoor.csv"
PERCLASS_CSV = "benchmark_yolov8_indoor_perclass.csv"
CLASS_NAMES = ["chair", "clock", "exit", "fireextinguisher", "printer", "screen", "trashbin"]

MODELS = [
    "yolov8n.pt",
    "yolov8s.pt",
    "yolov8m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


Best model: yolov10s — loading ../runs/detect/yolov10s_ood/weights/best.pt


<Figure size 1800x1800 with 9 Axes>

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path("../runs/detect/yolov8n_indoor4")
images_to_show = [
    "results.png",
    "confusion_matrix.png",
    "BoxF1_curve.png",
    "BoxPR_curve.png"
]

for img_name in images_to_show:
    img_path = results_dir / img_name
    if img_path.exists():
        plt.figure(figsize=(12, 10))
        img = mpimg.imread(str(img_path))
        plt.imshow(img)
        plt.axis('off')
        plt.title(img_name)
        plt.show()
    else:
        print(f"Warning: {img_path} not found.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
dir = Path("../runs/detect/yolov8n_indoor4")
for img in ["results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    p = dir / img
    if p.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(str(p)))
        plt.axis("off")
        plt.show()

In [ ]:
try:
    import pandas as pd
    print("Pandas OK")
except:
    !pip install pandas
    import pandas as pd

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df.to_string(index=False))
styled = df.style.highlight_max(subset=['mAP@0.5', 'neg_acc'], color='#2d6a2e')
display(styled)


# Indoor Dataset - Final Performance Benchmarks

## 1. Training Visualization

### 1.1 YOLOv8n Indoor (Nano)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

print("Displaying YOLOv8n Results...")
dir_n = Path("../runs/detect/yolov8n_indoor4")
images = ["results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"]
for img in images:
    p = dir_n / img
    if p.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(str(p)))
        plt.axis("off")
        plt.title(f"YOLOv8n Indoor - {img}")
        plt.show()

### 1.2 YOLOv8s Indoor (Small)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

print("Displaying YOLOv8s Results...")
dir_s = Path("../runs/detect/yolov8s_indoor")
images = ["results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"]
for img in images:
    p = dir_s / img
    if p.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(str(p)))
        plt.axis("off")
        plt.title(f"YOLOv8s Indoor - {img}")
        plt.show()

### 1.3 YOLOv8m Indoor (Medium)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

print("Displaying YOLOv8m Results...")
dir_m = Path("../runs/detect/yolov8m_indoor")
images = ["results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"]
for img in images:
    p = dir_m / img
    if p.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(str(p)))
        plt.axis("off")
        plt.title(f"YOLOv8m Indoor - {img}")
        plt.show()

## 2. Model Performance Benchmarks

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df.to_string(index=False))
styled = df.style.highlight_max(subset=['mAP@0.5', 'neg_acc'], color='#2d6a2e')
display(styled)
